# A1_Q3: Linear Regression with Regularization

* Using `student_data.csv` from Assignment 1 for dataset
* Task is to predict `GPA` using student features
* Need to use a validation set to select the hyperparameters

## Given code template:

* ```python
    def run_poly_regression(X_train, y_train, X_val, y_val, X_test, y_test, degree=1, regularizer=None, reg_strength=0.0):
        """
        Fit a polynomial regression model with optimal regularization.

        Parameters:
            degreee (int): Degree of the polynomial to fit
            regularizer (str or None): 'l1', 'l2', or None
            reg_strength (float): Regularization coefficient (alpha)

        Returns: dict with train, val, and test MSEs, and learned coefficients
        """
    ```

## Tasks to perform:

* For three setups - no regularization, L1 and L2 regularization, repeat the below steps:
    - Fit polynomial regression models across degrees 1 to 6.
    - Plot polynomial degree vs MSE (on train and validation sets). Describe the trend
    you observe as degree increases.
    - For each degree, use val MSE to choose the best regularization strength.
    - Plot regularization strength (log scale) vs val MSE for best degree.
* Comment on performance improvement (if any) from regularization. Which overall
experimental setup (degree, regularizer) yielded the best test performance?
* For the best setup using L1 regularization, which features had non-zero weights?
List the most important predictors for GPA. Repeat the same with L2 regularization.
Comment on the differences.
* You are allowed to use sklearn.linear model.LinearRegression

### Step 1: Data Preparation

In [45]:
import pandas as pd
import hashlib
from sklearn.model_selection import train_test_split

class DataPreprocessor:
    """
    A class to preprocess student data

    It encapsulated loading, feature/target separation, encoding,
    and splitting the data into training, validation, and test sets.
    """
    def __init__(self, filepath, target_column, username):
        self.filepath = filepath
        self.target_column = target_column
        self.username = username
        self.seed = self._generate_seed()

        # Initialize attributes to store data at various stages
        self.df = None
        self.X = None
        self.Y = None
        self.X_encoded = None
        self.X_train, self.y_train = None, None
        self.X_val, self.y_val = None, None
        self.X_test, self.y_test = None, None

    def _generate_seed(self):
        """Generates a consistent random seed from a username"""
        return int(hashlib.sha256(self.username.encode()).hexdigest(), 16) % (2**32)
    
    def load_data(self):
        """Loads data from the CSV file."""
        self.df = pd.read_csv(self.filepath)
        print(self.df.head())
        return self
    
    def separate_features_target(self):
        """Separates the Dataframe into features (X) and target (y)"""
        if self.df is None:
            raise ValueError("Data is not loaded. Please call load_data() first.")
        self.X = self.df.drop(columns=self.target_column)
        self.y = self.df[self.target_column]
        return self
    
    def encode_categorical_features(self, columns_to_encode):
        """Applies one-hot encoding to specified categorical columns."""
        if self.X is None:
            raise ValueError("Features (X) not created. Please call separate_features_target() first.")
        self.X_encoded = pd.get_dummies(self.X, columns=columns_to_encode, drop_first=True)
        print(self.X_encoded.head())
        return self
    
    def split_data(self, test_size=0.2, val_size=0.5):
        """Splits the data into training, validation, and test sets."""
        if self.X_encoded is None:
            raise ValueError("Data not encoded. Please call encode_categorical_features() first.")
        
        # Split into training (80%) and temporary (20%) sets
        self.X_train, X_temp, self.y_train, y_temp = train_test_split(
            self.X_encoded, self.y, test_size=test_size, random_state=self.seed
        )

        # Split the temporary set into validation (10%) and test (10%) sets
        self.X_val, self.X_test, self.y_val, self.y_test = train_test_split(
            X_temp, y_temp, test_size=val_size, random_state=self.seed
        )
        print(f"Training set size: {len(self.X_train)}, {len(self.y_train)} samples")
        print(f"Validation set size: {len(self.X_val)}, {len(self.y_val)} samples")
        print(f"Test set size: {len(self.X_test)}, {len(self.y_test)} samples")
        return self
        
    def get_split_data(self):
        """Returns the final split datasets."""
        return self.X_train, self.X_val, self.X_test, self.y_train, self.y_val, self.y_test

In [46]:
# Define parameters
FILEPATH = 'student_data.csv'
TARGET = 'GPA'
USERNAME = 'bhogadi.likhit'
CATEGORICAL_COLS = ['Gender', 'Major', 'Program']

In [47]:
# 1. Instantiate the preprocessor
preprocessor = DataPreprocessor(filepath=FILEPATH, target_column=TARGET, username=USERNAME)

In [48]:
# 2. Run the preprocessing steps in a chain
(preprocessor.load_data()
             .separate_features_target()
             .encode_categorical_features(columns_to_encode=CATEGORICAL_COLS)
             .split_data())

   Gender   Major Program   GPA
0    Male      MS     CSE  7.59
1    Male  B.Tech     CHD  5.19
2    Male      MS     ECE  8.05
3    Male      MS     ECE  7.35
4  Female      MS     CND  7.87
   Gender_Male  Gender_Other  Major_MS  Major_PhD  Program_CND  Program_CSE  \
0         True         False      True      False        False         True   
1         True         False     False      False        False        False   
2         True         False      True      False        False        False   
3         True         False      True      False        False        False   
4        False         False      True      False         True        False   

   Program_ECE  
0        False  
1        False  
2         True  
3         True  
4        False  
Training set size: 8000, 8000 samples
Validation set size: 1000, 1000 samples
Test set size: 1000, 1000 samples


In [49]:
# 3. Retrieve the final datasets
X_train, X_val, X_test, y_train, y_val, y_test = preprocessor.get_split_data()

### Step 2: Build the Regression Pipeline

In [60]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Lasso, Ridge

def build_regression_pipeline(degree=1, regularizer=None, reg_strength=0.0):
    """
    Creates a machine learning pipeline for polynomial regression with optional regularization.
    Parameters:
    - degree: Degree of the polynomial features.
    - regularizer: Type of regularization ('l1' for Lasso, 'l2' for Ridge, None for no regularization).
    - reg_strength: Regularization strength (alpha).
    Returns:
    - A sklearn Pipeline object.
    """
    # A list of (name, transformer/estimator) tuples
    steps = [
        ('poly_features', PolynomialFeatures(degree=degree, include_bias=False)),  # First step: polynomial features
        ('scaler', StandardScaler()),         # Second step: a transformer
    ]

    if regularizer == 'l1':
        steps.append(('regressor', Lasso(alpha=reg_strength)))
    elif regularizer == 'l2':
        steps.append(('regressor', Ridge(alpha=reg_strength)))
    elif regularizer is None:
        steps.append(('regressor', LinearRegression()))
    else:
        raise ValueError("regularizer must be 'l1', 'l2', or None")

    # Create the pipeline object
    pipeline = Pipeline(steps)

    return pipeline

### Step 3: The Experiment Loop

In [62]:
import numpy as np

def run_poly_regression(X_train, y_train, X_val, y_val, X_test, y_test):
    """
    Trains polynomial regression models with varying degrees and regularization,
    evaluates them, and stores the results.
    Parameters:
    - X_train, y_train: Training data
    - X_val, y_val: Validation data
    - X_test, y_test: Test data
    Returns:
    - A list of dictionaries containing results for each configuration.
    """
    resuts = [] # To store results for each configuration

    for regularizer in ['None', 'l1', 'l2']:
        for degree in range(1, 7): # Degrees 1 to 6
            
            # If using regularization, you need to find the best strength
            if regularizer in ['l1', 'l2']:
                best_val_mse = float('inf')
                best_strength = None
                
                # A good range for regularization strength on a log scale
                strengths_to_try = np.logspace(-4, 2, 20) 
                
                for strength in strengths_to_try:
                    # 1. Create your pipeline (PolynomialFeatures, StandardScaler, Model)
                    #    - Set the model to Lasso(alpha=strength) or Ridge(alpha=strength)
                    pipeline = build_regression_pipeline(degree=degree, regularizer=regularizer, reg_strength=strength)

                    # 2. Fit the pipeline on the TRAINING data
                    pipeline.fit(X_train, y_train)

                    # 3. Calculate MSE on the VALIDATION data
                    y_val_pred = pipeline.predict(X_val)
                    current_val_mse = np.mean((y_val - y_val_pred) ** 2)
                    
                    # If this strength is the best so far, save it
                    if current_val_mse < best_val_mse:
                        best_val_mse = current_val_mse
                        best_strength = strength
                
                # Now you have the best strength for this degree.
                # Re-train a model with this best strength to get the final
                best_strength_pipeline = build_regression_pipeline(degree=degree, regularizer=regularizer, reg_strength=best_strength)
                best_strength_pipeline.fit(X_train, y_train)

                # train/val/test scores for this degree.
                y_train_pred = best_strength_pipeline.predict(X_train)
                y_val_pred = best_strength_pipeline.predict(X_val)
                y_test_pred = best_strength_pipeline.predict(X_test)
                train_mse = np.mean((y_train - y_train_pred) ** 2)
                val_mse = np.mean((y_val - y_val_pred) ** 2)
                test_mse = np.mean((y_test - y_test_pred) ** 2)

            else: # No regularization
                # 1. Create your pipeline (PolynomialFeatures, StandardScaler, LinearRegression)
                pipeline = build_regression_pipeline(degree=degree, regularizer=None)

                # 2. Fit on TRAINING data
                pipeline.fit(X_train, y_train)
                # 3. Calculate and store MSE for TRAIN, VALIDATION, and TEST sets.
                y_train_pred = pipeline.predict(X_train)
                y_val_pred = pipeline.predict(X_val)
                y_test_pred = pipeline.predict(X_test)
                train_mse = np.mean((y_train - y_train_pred) ** 2)
                val_mse = np.mean((y_val - y_val_pred) ** 2)
                test_mse = np.mean((y_test - y_test_pred) ** 2)
            
            # Store all results (degree, best_strength, train_mse, val_mse, test_mse)
            # for later plotting and analysis.
            resuts.append({
                'regularizer': regularizer,
                'degree': degree,
                'best_strength': best_strength if regularizer in ['l1', 'l2'] else None,
                'train_mse': train_mse,
                'val_mse': val_mse,
                'test_mse': test_mse
            })

    return resuts